In [1]:
import pandas as pd
import numpy as np
import re
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
df = pd.read_csv("marketplace_dataset.csv")

In [4]:
df.head()

,id,title,description,price,category,image,user_id,is_sold,buyer_id,created_at
0,1,graphics kit,gently used with no scratches,1000.0,Others,160d2d9d090145b4bba2558c1970ca04.jpg,1,True,2.0,NaN
1,2,Engineering Mathematics 2,2025 version gently used,250.0,Books,557e2b24d71044dd85928050b187eba7.png,1,False,NaN,NaN
2,3,Redgear Mechanical Keyboard,Brand: Redgear\nCondition: Excellent\n\nUsed c...,1496.0,Electronics,default.jpg,5,False,NaN,2026-05-18 07:47:11.808560
3,4,Electronic Component Kit - Like New,Brand: Generic\nCondition: Fair\n\nEverything ...,453.0,Stationery and Tools,default.jpg,3,False,NaN,2026-06-26 10:03:11.809124
4,5,Precision Screwdriver Set - Like New,Brand: Taparia\nCondition: Excellent\n\nEveryt...,578.0,Stationery and Tools,default.jpg,1,True,4.0,2026-04-07 21:22:11.809350


In [5]:
df.shape

(522, 10)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 522 entries, 0 to 521
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           522 non-null    int64  
 1   title        491 non-null    object 
 2   description  522 non-null    object 
 3   price        522 non-null    float64
 4   category     522 non-null    object 
 5   image        522 non-null    object 
 6   user_id      522 non-null    int64  
 7   is_sold      522 non-null    bool   
 8   buyer_id     79 non-null     float64
 9   created_at   520 non-null    object 
dtypes: bool(1), float64(2), int64(2), object(5)
memory usage: 37.3+ KB


In [7]:
df.describe()

,id,price,user_id,buyer_id
count,522.000000,522.000000,522.000000,79.000000
mean,261.500000,1949.720307,4.333333,4.405063
std,150.832689,4959.418750,2.271411,2.278770
min,1.000000,56.000000,1.000000,1.000000
25%,131.250000,300.750000,2.000000,2.000000
50%,261.500000,577.000000,4.000000,5.000000
75%,391.750000,1203.500000,6.000000,6.000000
max,522.000000,36290.000000,8.000000,8.000000


In [8]:
df = df[
    [
        "id",
        "title",
        "description",
        "category",
        "price",
        "is_sold"
    ]
]

In [9]:
df.isnull().sum()

id              0
title          31
description     0
category        0
price           0
is_sold         0
dtype: int64

In [10]:
df["title"] = df["title"].fillna("")

In [11]:
df.isnull().sum()

id             0
title          0
description    0
category       0
price          0
is_sold        0
dtype: int64

In [12]:
df["combined_text"] = (
    df["title"]
    + " "
    + df["category"]
    + " "
    + df["description"]
)

In [13]:
df[
    [
        "title",
        "combined_text"
    ]
].head()

,title,combined_text
0,graphics kit,graphics kit Others gently used with no scratches
1,Engineering Mathematics 2,Engineering Mathematics 2 Books 2025 version g...
2,Redgear Mechanical Keyboard,Redgear Mechanical Keyboard Electronics Brand:...
3,Electronic Component Kit - Like New,Electronic Component Kit - Like New Stationery...
4,Precision Screwdriver Set - Like New,Precision Screwdriver Set - Like New Stationer...


In [59]:
def clean_text(text):
    text = text.lower()

    text = re.sub(r'[^a-zA-Z0-9 ]', ' ', text)

    words = text.split()

    remove_words = {
        "excellent",
        "good",
        "condition",
        "sale",
        "selling",
        "used",
        "new",
        "like",
        "well",
        "ready"
    }

    words = [word for word in words if word not in remove_words]

    text = " ".join(words)

    text = re.sub(r'\s+', ' ', text)

    return text.strip()


def remove_duplicate_words(text):
    words = text.split()
    return " ".join(dict.fromkeys(words))

In [60]:
df["combined_text"] = df["combined_text"].apply(clean_text)
df["combined_text"] = df["combined_text"].apply(remove_duplicate_words)

In [61]:
df[
    [
        "combined_text"
    ]
].head()

,combined_text
0,graphics kit others gently with no scratches
1,engineering mathematics 2 books 2025 version g...
2,redgear mechanical keyboard electronics brand ...
3,electronic component kit stationery and tools ...
4,precision screwdriver set stationery and tools...


In [62]:
df[df["combined_text"] == ""]

,id,title,description,category,price,is_sold,combined_text


In [63]:
df.head()

,id,title,description,category,price,is_sold,combined_text
0,1,graphics kit,gently used with no scratches,Others,1000.0,True,graphics kit others gently with no scratches
1,2,Engineering Mathematics 2,2025 version gently used,Books,250.0,False,engineering mathematics 2 books 2025 version g...
2,3,Redgear Mechanical Keyboard,Brand: Redgear\nCondition: Excellent\n\nUsed c...,Electronics,1496.0,False,redgear mechanical keyboard electronics brand ...
3,4,Electronic Component Kit - Like New,Brand: Generic\nCondition: Fair\n\nEverything ...,Stationery and Tools,453.0,False,electronic component kit stationery and tools ...
4,5,Precision Screwdriver Set - Like New,Brand: Taparia\nCondition: Excellent\n\nEveryt...,Stationery and Tools,578.0,True,precision screwdriver set stationery and tools...


In [64]:



custom_stop_words = {
    "excellent",
    "good",
    "condition",
    "sale",
    "selling",
    "used",
    "new",
    "like",
    "well",
    "ready"
}

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=1000,
    ngram_range=(1,2)
)


In [65]:
tfidf_matrix = vectorizer.fit_transform(df["combined_text"])

In [66]:
tfidf_matrix.shape

(522, 747)

In [67]:
vectorizer.get_feature_names_out()[:50]

array(['15', '15 electronics', '20000mah', '20000mah electronics', '2025',
       '2025 version', '24', '24 inch', '3511', '3511 electronics', '3m',
       '3m maintained', '3m project', '450', '450 electronics', '512gb',
       '512gb electronics', '991es', '991es plus', 'anymore',
       'anymore reason', 'arduino', 'arduino uno', 'asked',
       'asked previous', 'backpack', 'backpack brand', 'backup',
       'backup performance', 'badminton', 'badminton racket', 'bank',
       'bank 20000mah', 'basket', 'basket hostel', 'basketball',
       'basketball sports', 'bat', 'bat sports', 'battery',
       'battery backup', 'beginners', 'beginners college', 'board',
       'board hostel', 'board stationery', 'boat', 'boat rockerz', 'book',
       'book longer'], dtype=object)

In [68]:
df["combined_text"].iloc[0]


'graphics kit others gently with no scratches'

In [69]:
df["category"].value_counts()

category
Stationery and Tools    120
Books                    79
Electronics              70
Sports                   50
Others                   47
Furniture                46
Notes                    44
Hostel Essentials        39
Cycles                   27
Name: count, dtype: int64

In [70]:
"and" in vectorizer.get_stop_words()

True

In [58]:
print(vectorizer.stop_words)

english


In [71]:
similarity_matrix = cosine_similarity(tfidf_matrix)

In [72]:
similarity_matrix.shape

(522, 522)

In [73]:
similarity_matrix

array([[1.        , 0.12124537, 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.12124537, 1.        , 0.        , ..., 0.02355691, 0.02756326,
        0.        ],
       [0.        , 0.        , 1.        , ..., 0.01832303, 0.02143925,
        0.0589598 ],
       ...,
       [0.        , 0.02355691, 0.01832303, ..., 1.        , 0.12982326,
        0.02305369],
       [0.        , 0.02756326, 0.02143925, ..., 0.12982326, 1.        ,
        0.02697446],
       [0.        , 0.        , 0.0589598 , ..., 0.02305369, 0.02697446,
        1.        ]], shape=(522, 522))

In [80]:
product_index = 67

scores = list(enumerate(similarity_matrix[product_index]))

scores[:10]

[(0, np.float64(0.0)),
 (1, np.float64(0.039403435584436965)),
 (2, np.float64(0.017047060242836493)),
 (3, np.float64(0.09574659024223241)),
 (4, np.float64(0.08977781347319395)),
 (5, np.float64(0.024293902439466886)),
 (6, np.float64(0.22262711569203159)),
 (7, np.float64(0.023231690778923522)),
 (8, np.float64(0.022697106035568734)),
 (9, np.float64(0.12718408237553416))]

In [81]:
sorted_scores = sorted(
    scores,
    key=lambda x: x[1],
    reverse=True
)

In [82]:
sorted_scores = sorted_scores[1:21]

In [83]:
for index, score in sorted_scores[:5]:
    print(df.iloc[index]["title"])
    print("Similarity:", round(score,3))
    print("-"*40)

Digital Multimeter
Similarity: 0.584
----------------------------------------
Digital Multimeter - Like New
Similarity: 0.574
----------------------------------------
Used Digital Multimeter
Similarity: 0.535
----------------------------------------
Digital Multimeter - Like New
Similarity: 0.526
----------------------------------------
Meco Digital Multimeter
Similarity: 0.5
----------------------------------------


In [78]:
print(df.iloc[0][["title", "category", "description"]])

print(df["title"].nunique())

print(df["category"].value_counts())

title                           graphics kit
category                              Others
description    gently used with no scratches
Name: 0, dtype: object
259
category
Stationery and Tools    120
Books                    79
Electronics              70
Sports                   50
Others                   47
Furniture                46
Notes                    44
Hostel Essentials        39
Cycles                   27
Name: count, dtype: int64


In [79]:
print(df.iloc[10]["combined_text"])

books brand generic original book in because i no longer need it reason for shifting to another city price slightly negotiable
